# bench_gen — Genomics Drug Sensitivity Benchmark with Semi-Supervised Tabular Models

This notebook benchmarks genomic drug response models on the GDSC dataset and
introduces a semi-supervised fine-tuning loop that reuses the shared
`BenchmarkRunner`, tabular registries, and SSL helpers.

- **Primary dataset:** [`samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc`](https://www.kaggle.com/datasets/samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc)
- **Secondary dataset (hold-out test):** [`crawford/ctrp-v2-cell-line-sensitivity`](https://www.kaggle.com/datasets/crawford/ctrp-v2-cell-line-sensitivity)

The pipeline casts IC50 prediction as a binary classification task (sensitive vs
resistant per drug) to leverage the existing classification utilities. You can
swap in regression heads if you prefer continuous metrics.


## 1. Imports and setup
We rely heavily on the shared tabular infrastructure: the model registry,
`BenchmarkRunner`, and the semi-supervised wrapper for tabular networks.


In [ ]:
import os
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import torch
from torch.utils.data import TensorDataset, DataLoader

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.ss_models import SemiSupervisedTabular
from pipelines_torch.base import SimplePredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print(f"Device: {DEVICE}")


In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


### Download datasets (optional Kaggle helper)
Place your Kaggle credentials in `~/.kaggle/kaggle.json` or pre-download the
archives into `data_gdsc/primary` and `data_gdsc/secondary`.


In [ ]:
DATA_ROOT = Path("data_gdsc")
PRIMARY_DIR = DATA_ROOT / "primary"
SECONDARY_DIR = DATA_ROOT / "secondary"
PRIMARY_SLUG = "samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc"
SECONDARY_SLUG = "crawford/ctrp-v2-cell-line-sensitivity"

for directory in (PRIMARY_DIR, SECONDARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    if not any(PRIMARY_DIR.iterdir()):
        print(f"Downloading {PRIMARY_SLUG} …")
        api.dataset_download_files(PRIMARY_SLUG, path=PRIMARY_DIR, unzip=True)
    if not any(SECONDARY_DIR.iterdir()):
        print(f"Downloading {SECONDARY_SLUG} …")
        api.dataset_download_files(SECONDARY_SLUG, path=SECONDARY_DIR, unzip=True)
except Exception as exc:  # pragma: no cover - depends on runtime
    print(f"Kaggle download skipped or failed: {exc}")


## 2. Data loading helpers
The loaders search for expression and drug-response files inside each directory.
We normalise cell line identifiers, restrict to the top drugs by coverage, and
convert IC50 values into drug-specific binary labels (sensitive vs resistant).


In [ ]:
def _find_file(root: Path, keywords: Tuple[str, ...]) -> Optional[Path]:
    for path in root.rglob("*.csv"):
        lower = path.name.lower()
        if all(keyword in lower for keyword in keywords):
            return path
    return None


def _load_expression_and_response(root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    expr_path = _find_file(root, ("expression",)) or _find_file(root, ("rnaseq",))
    resp_path = _find_file(root, ("response",)) or _find_file(root, ("ic50",))
    if expr_path is None or resp_path is None:
        raise FileNotFoundError(
            f"Could not locate expression/response CSVs under {root}."
            " Please inspect the dataset and adjust the keywords."
        )
    expr = pd.read_csv(expr_path)
    resp = pd.read_csv(resp_path)
    return expr, resp


def _detect_columns(resp: pd.DataFrame) -> Tuple[str, str, str]:
    lower_cols = {c.lower(): c for c in resp.columns}
    cell_col = next((col for key, col in lower_cols.items() if "cell" in key and "id" in key), None)
    if cell_col is None:
        cell_col = next((col for key, col in lower_cols.items() if "cell" in key), None)
    drug_col = next((col for key, col in lower_cols.items() if "drug" in key or "compound" in key), None)
    ic50_col = next((col for key, col in lower_cols.items() if "ic50" in key or "auc" in key), None)
    if not all([cell_col, drug_col, ic50_col]):
        raise KeyError("Could not infer cell/drug/IC50 columns from response table")
    return cell_col, drug_col, ic50_col


def prepare_dataset(
    root: Path,
    gene_columns: Optional[List[str]] = None,
    drug_levels: Optional[List[str]] = None,
    top_k_drugs: int = 5,
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    expr_raw, resp_raw = _load_expression_and_response(root)
    expr = expr_raw.set_index(expr_raw.columns[0])
    expr = expr.select_dtypes(include=[np.number])
    expr.index = expr.index.astype(str).str.upper()

    resp = resp_raw.copy()
    cell_col, drug_col, ic50_col = _detect_columns(resp)
    resp = resp.dropna(subset=[cell_col, drug_col, ic50_col])
    resp[cell_col] = resp[cell_col].astype(str).str.upper()
    resp = resp[resp[cell_col].isin(expr.index)].reset_index(drop=True)
    resp["log_ic50"] = np.log1p(resp[ic50_col].astype(float))

    if gene_columns is None:
        gene_columns = expr.columns.tolist()
    if drug_levels is None:
        drug_levels = resp[drug_col].value_counts().nlargest(top_k_drugs).index.tolist()

    resp = resp[resp[drug_col].isin(drug_levels)].reset_index(drop=True)
    if resp.empty:
        raise ValueError(f"No overlapping drugs found in {root} for {drug_levels}")

    # Build feature matrix with consistent gene columns
    expr_aligned = expr.reindex(columns=gene_columns, fill_value=0.0)
    expr_slice = expr_aligned.loc[resp[cell_col]].reset_index(drop=True)

    drug_dummies = pd.get_dummies(resp[drug_col], prefix="drug")
    for level in drug_levels:
        col = f"drug_{level}"
        if col not in drug_dummies.columns:
            drug_dummies[col] = 0
    drug_dummies = drug_dummies[[f"drug_{level}" for level in drug_levels]].reset_index(drop=True)

    features_df = pd.concat([expr_slice.reset_index(drop=True), drug_dummies], axis=1)

    # Binary label per drug (lower IC50 => sensitive)
    resp["label"] = resp.groupby(drug_col)["log_ic50"].transform(lambda s: (s <= s.median()).astype(int))
    labels = resp["label"].to_numpy(dtype=np.int64)

    return features_df.to_numpy(dtype=np.float32), labels, gene_columns, drug_levels


### Prepare train/validation and external test splits
We keep 20% of the primary dataset as a validation set and reserve the secondary
collection for out-of-domain testing.


In [ ]:
X_primary, y_primary, gene_columns, drug_levels = prepare_dataset(PRIMARY_DIR)
X_train, X_val, y_train, y_val = train_test_split(
    X_primary,
    y_primary,
    test_size=0.2,
    random_state=SEED,
    stratify=y_primary,
)

try:
    X_secondary, y_secondary, _, _ = prepare_dataset(SECONDARY_DIR, gene_columns=gene_columns, drug_levels=drug_levels)
except Exception as exc:
    print(f"Secondary dataset warning: {exc}
Using validation metrics only.")
    X_secondary, y_secondary = None, None

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_all_scaled = scaler.transform(X_primary)
X_secondary_scaled = scaler.transform(X_secondary) if X_secondary is not None else None


## 3. Baseline benchmark with `BenchmarkRunner`
We compare a shallow and a deeper MLP from the registry. Add additional keys
(e.g. `tabr_classifier`) as desired.


In [ ]:
def accuracy_metric(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return accuracy_score(y_true, preds)


def f1_macro_metric(y_true, probs):
    preds = np.argmax(probs, axis=1)
    return f1_score(y_true, preds, average="macro")


def roc_auc_metric(y_true, probs):
    try:
        return roc_auc_score(y_true, probs[:, 1])
    except Exception:
        return float("nan")

accuracy_metric.__name__ = "accuracy"
f1_macro_metric.__name__ = "f1_macro"
roc_auc_metric.__name__ = "roc_auc"
metrics = [accuracy_metric, f1_macro_metric, roc_auc_metric]


In [ ]:
input_dim = X_train_scaled.shape[1]
model_configs = [
    {
        "name": "mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {
            "input_dim": input_dim,
            "hidden_dims": [512, 256, 128],
            "num_classes": 2,
            "dropout": 0.3,
            "batchnorm": True,
        },
    },
    {
        "name": "deep_mlp_classifier",
        "class": MODEL_REGISTRY["deep_mlp_classifier"],
        "params": {
            "input_dim": input_dim,
            "num_classes": 2,
        },
    },
]


In [ ]:
runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    metrics=metrics,
    task_type="classification",
    device=DEVICE,
    epochs=12,
    batch_size=64,
    learning_rate=3e-4,
    weight_decay=1e-4,
    use_class_weights=True,
    use_kfold=False,
    random_state=SEED,
    path_start="bench_gen_supervised",
)
results_supervised = runner.run(X_train_scaled, y_train)
results_supervised.sort_values("score", ascending=False)


### Evaluate saved models on validation / external sets
We reload the checkpoints created by `BenchmarkRunner` to measure validation and
external performance consistently.


In [ ]:
def evaluate_models(model_names: List[str], X: np.ndarray, y: np.ndarray, results_dir: str) -> pd.DataFrame:
    records = []
    for name in model_names:
        ckpt = f"{name}_none"
        try:
            model = MODEL_REGISTRY[name](**model_configs[[cfg["name"] for cfg in model_configs].index(name)]["params"])
            state_path = Path("results") / results_dir / f"{ckpt}.pt"
            if not state_path.exists():
                print(f"⚠️ Missing checkpoint for {name} at {state_path}")
                continue
            state_dict = torch.load(state_path, map_location=DEVICE)
            model.load_state_dict(state_dict)
        except Exception as exc:
            print(f"Skipping {name} due to load error: {exc}")
            continue
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=128)
        probs = predictor.predict_proba(X)
        records.append({
            "model": name,
            "accuracy": accuracy_metric(y, probs),
            "f1_macro": f1_macro_metric(y, probs),
            "roc_auc": roc_auc_metric(y, probs),
        })
    return pd.DataFrame.from_records(records)

val_metrics = evaluate_models([cfg["name"] for cfg in model_configs], X_val_scaled, y_val, "bench_gen_supervised")
val_metrics


In [ ]:
if X_secondary_scaled is not None:
    external_metrics = evaluate_models([cfg["name"] for cfg in model_configs], X_secondary_scaled, y_secondary, "bench_gen_supervised")
    display(external_metrics)


## 4. Semi-supervised fine-tuning with `SemiSupervisedTabular`
We reuse the registry MLP and the shared MeanTeacher implementation to exploit
unlabeled samples (a random 40% of training rows) with confidence-based updates.


In [ ]:
def train_tabular_ssl(
    X: np.ndarray,
    y: np.ndarray,
    *,
    unlabeled_fraction: float = 0.4,
    epochs: int = 20,
    batch_size: int = 128,
    lr: float = 3e-4,
) -> Tuple[SemiSupervisedTabular, pd.DataFrame]:
    input_dim = X.shape[1]
    base_model = MODEL_REGISTRY["mlp_classifier"](
        input_dim=input_dim,
        hidden_dims=[512, 256, 128],
        num_classes=2,
        dropout=0.3,
        batchnorm=True,
    )
    ssl_model = SemiSupervisedTabular(base_model, num_classes=2, use_mean_teacher=True).to(DEVICE)
    optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=lr, weight_decay=1e-4)

    rng = np.random.default_rng(SEED)
    mask = rng.random(len(y)) > unlabeled_fraction
    labeled_idx = np.where(mask)[0]
    unlabeled_idx = np.where(~mask)[0]

    labeled_ds = TensorDataset(
        torch.tensor(X[labeled_idx], dtype=torch.float32),
        torch.tensor(y[labeled_idx], dtype=torch.long),
    )
    unlabeled_ds = TensorDataset(
        torch.tensor(X[unlabeled_idx], dtype=torch.float32),
        torch.zeros(len(unlabeled_idx)),
    )

    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=batch_size * 2, shuffle=True, drop_last=True)

    history = []
    unlabeled_iter = cycle(unlabeled_loader) if len(unlabeled_ds) > 0 else None

    for epoch in range(epochs):
        ssl_model.train()
        total_loss = 0.0
        steps = 0
        if unlabeled_iter is None:
            break
        for xb_l, yb_l in labeled_loader:
            xb_u, _ = next(unlabeled_iter)
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)

            optimizer.zero_grad()
            loss, logs = ssl_model.step((xb_l, yb_l), (xb_u, None), epoch)
            loss.backward()
            optimizer.step()
            ssl_model.post_step()

            total_loss += loss.item()
            steps += 1

        ssl_model.eval()
        with torch.no_grad():
            logits = ssl_model(torch.tensor(X_val_scaled, dtype=torch.float32, device=DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            val_acc = accuracy_score(y_val, preds)
            val_f1 = f1_score(y_val, preds, average="macro")
        history.append({"epoch": epoch + 1, "loss": total_loss / max(1, steps), "val_accuracy": val_acc, "val_f1": val_f1})
        print(f"Epoch {epoch+1:02d}: loss={history[-1]['loss']:.4f} val_f1={val_f1:.3f}")

    return ssl_model, pd.DataFrame(history)


In [ ]:
ssl_model, ssl_history = train_tabular_ssl(X_all_scaled, y_primary, epochs=15, unlabeled_fraction=0.4)
ssl_history


In [ ]:
def evaluate_ssl(model: torch.nn.Module, X: np.ndarray, y: np.ndarray) -> pd.Series:
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32, device=DEVICE))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    return pd.Series({
        "accuracy": accuracy_score(y, preds),
        "f1_macro": f1_score(y, preds, average="macro"),
        "roc_auc": roc_auc_metric(y, probs),
    })

ssl_val_metrics = evaluate_ssl(ssl_model, X_val_scaled, y_val)
print("SSL validation metrics:
", ssl_val_metrics)
if X_secondary_scaled is not None:
    ssl_ext_metrics = evaluate_ssl(ssl_model, X_secondary_scaled, y_secondary)
    print("
SSL external metrics:
", ssl_ext_metrics)


### Notes & extensions
- Adjust `top_k_drugs` or the binary label thresholding strategy to suit your
  downstream evaluation.
- Swap the registry model (e.g. use `tabr_classifier`) or plug in a regression
  head by adapting the target definition.
- Integrate additional unlabeled cohorts by concatenating their feature matrices
  before invoking the SSL routine.
